# JLab Hackathon 2021 Problem02

Here is my attempt to solve Problem02 from the hackathon. 

First tasks are to mount Google drive and to download the data.

Note that I don't download the data to the Google drive (I download to the temporary work space on Colab). I mount the drive here for later use but because it requires logging in and copying  a URL that blocks the cell from completing, I want it done early so I can walk away and let the rest of the notebook run if I need to re-run the whole thing.

In [1]:
#from google.colab import drive
#drive.mount('/gdrive')
!ls
#!mkdir -p /gdrive/MyDrive/work/2021.07.29.hackathon

!wget https://userweb.jlab.org/~tbritton/Hackathon2021_DataSets/Problem2/train_events.csv
!wget https://userweb.jlab.org/~tbritton/Hackathon2021_DataSets/Problem2/test_events.csv
!wget https://userweb.jlab.org/~tbritton/Hackathon2021_DataSets/Problem2/judge_events.csv
!ls -l

Hackathon2021_Problem02.ipynb  prob02_model07	  train_events.csv
judge_events.csv	       test_events.csv	  train_events.csv.1
judge_events.csv.1	       test_events.csv.1
--2021-08-05 18:28:38--  https://userweb.jlab.org/~tbritton/Hackathon2021_DataSets/Problem2/train_events.csv
Resolving userweb.jlab.org (userweb.jlab.org)... 129.57.64.126
Connecting to userweb.jlab.org (userweb.jlab.org)|129.57.64.126|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 450000000 (429M) [text/csv]
Saving to: ‘train_events.csv.2’

train_events.csv.2  100%[===================>] 429.15M   100MB/s    in 4.6s    

2021-08-05 18:28:43 (93.1 MB/s) - ‘train_events.csv.2’ saved [450000000/450000000]

--2021-08-05 18:28:43--  https://userweb.jlab.org/~tbritton/Hackathon2021_DataSets/Problem2/test_events.csv
Resolving userweb.jlab.org (userweb.jlab.org)... 129.57.64.126
Connecting to userweb.jlab.org (userweb.jlab.org)|129.57.64.126|:443... connected.
HTTP request sent, awaiting response... 

### Read in data

I define a single procedure here so it can be used for all 3 data sets (train, test, and judge).

In [2]:
import pandas as pd
import numpy as np

# Maximum amplitude from examination of data outside of this notebook is 1.
amp_max = 1.0

# Define procedure for reading in an rescaling the data so it can be used
# for train, test, and judge datasets
def ReadDataset( csv_fname ):
  df = pd.read_csv(csv_fname, header=None)
  arr = np.array(df)
  x = arr[:,:900]
  y = arr[:,900:]
  print( 'Read %s:  x.shape=%s  y.shape=%s' % ( csv_fname, str(x.shape), str(y.shape)) )

  # Scale both the inputs and the outputs to be 0-1
  x = x/amp_max
  y = y/amp_max

  x = x.reshape(arr.shape[0],30,30,1)
  if arr.shape[1] > 900:
    y = y.reshape(arr.shape[0],30,30,1)
    print('amp_max: %f  min(x): %f  max(x):  %f  min(y): %f  max(y):  %f' % (amp_max, np.min(x), np.max(x), np.min(y), np.max(y) ))
  else:
    print('amp_max: %f  min(x): %f  max(x):  %f ' % (amp_max, np.min(x), np.max(x) ))

  return (x, y)

# Read in all 3 data sets
(xtrain, ytrain) = ReadDataset('train_events.csv')
(xtest,  ytest ) = ReadDataset('test_events.csv')
(xjudge, yjudge) = ReadDataset('judge_events.csv')


Read train_events.csv:  x.shape=(10000, 900)  y.shape=(10000, 900)
amp_max: 1.000000  min(x): 0.000000  max(x):  1.000000  min(y): 0.000000  max(y):  0.988235
Read test_events.csv:  x.shape=(2000, 900)  y.shape=(2000, 900)
amp_max: 1.000000  min(x): 0.000000  max(x):  0.941176  min(y): 0.000000  max(y):  0.941176
Read judge_events.csv:  x.shape=(2000, 900)  y.shape=(2000, 0)
amp_max: 1.000000  min(x): 0.000000  max(x):  0.933333 


### Create model

In [11]:
import os
import tensorflow.keras as keras
import tensorflow.keras.backend as K

# Filename for saving model to. This also serves as a check to skip
# building and fitting the model again so we can re-run this whole
# notebook without having to re-train.
model_fname = 'prob02_model07'

# This is a custom metric to estimate the final solution score for each batch.
# It is useful for monitoring during training, especially if tweaking the 
# model and I want to know right away how good it is doing. 
def EstimatedScore(y_true, y_pred):
  ydiff = (y_pred - y_true)*amp_max
  return K.sum( ydiff*ydiff )*2000/y_true.shape[0]

if not os.path.exists(model_fname) or True:

  # My first thought was to build a model with a small latent space
  # to try and force the image's essence to be trained there. My
  # second thought is whether this is really necessary.
  #model = keras.Sequential()
  
  inputs        = keras.layers.Input(shape=(30,30,1))
  convLayer     = keras.layers.Conv2D(128, (5,5), name='convLayer')(inputs)
  convLayer2    = keras.layers.Conv2D(128, (5,5), name='convLayer2')(convLayer)
  #convLayer3    = keras.layers.Conv2D(128, (3,3), name='convLayer3')(convLayer2)
  flatInputs    = keras.layers.Flatten()(inputs)
  flatConv      = keras.layers.Flatten()(convLayer2)
  combinedLayer = keras.layers.Concatenate()([flatInputs, flatConv])

  # x = keras.layers.Dense(900, activation='linear')(combinedLayer)
  # x = keras.layers.Dense(768, activation='linear')(x)
  # x = keras.layers.Dense(512, activation='selu')(x)
  # x = keras.layers.Dense(256, activation='linear')(x)
  # x = keras.layers.Dense(512, activation='selu')(x)
  # x = keras.layers.Dense(768, activation='linear')(x)
  # x = keras.layers.Dense(900, activation='linear')(x)

  x = keras.layers.Dense(900, activation='linear')(combinedLayer)
  x = keras.layers.Dense(900, activation='selu')(x)
  x = keras.layers.Dense(900, activation='linear')(x)
  x = keras.layers.Dropout(0.05)(x)
  x = keras.layers.Dense(900, activation='selu')(x)
  x = keras.layers.Dense(900, activation='linear')(x)
  x = keras.layers.Dropout(0.05)(x)
  # x = keras.layers.Dense(900, activation='selu')(x)
  # x = keras.layers.Dense(900, activation='linear')(x)
  # x = keras.layers.Dropout(0.05)(x)
  x = keras.layers.Dense(900, activation='selu')(x)
  x = keras.layers.Dense(900, activation='linear')(x)
  x = keras.layers.Dense(900, activation='tanh')(x)
  x = keras.layers.Dense(900, activation='linear')(x)
  # x = keras.layers.Dense(900, activation='selu')(x)
  # x = keras.layers.Dense(900, activation='linear')(x)
  # x = keras.layers.Dense(900, activation='selu')(x)
  # x = keras.layers.Dense(900, activation='linear')(x)

  outputs = keras.layers.Reshape((30,30,1))(x)

  model = keras.Model(inputs=inputs, outputs=outputs)

  model.compile(optimizer='adam', loss='MSE', metrics=EstimatedScore)
  model.summary()

  early_stop = keras.callbacks.EarlyStopping(monitor='val_loss',
                                             patience=20,
                                             mode='min',
                                             min_delta=1.0E-8,
                                             restore_best_weights=True)

  model.fit(xtrain, ytrain, epochs=1000, callbacks=[early_stop], batch_size=1000, validation_data=(xtest, ytest))
  #model.fit(xtrain, ytrain, epochs=1000, callbacks=[early_stop], batch_size=100, validation_data=(xtest, ytest))
  #model.fit(xtrain, ytrain, epochs=1000, callbacks=[early_stop], batch_size=25, validation_data=(xtest, ytest))

  model.save( model_fname )

else:
  print('Model: %s already exists. Skipping training and loading from saved ...' % model_fname)
  model = keras.models.load_model(model_fname, custom_objects={'EstimatedScore':EstimatedScore})
  model.summary()

Model: "functional_7"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_4 (InputLayer)            [(None, 30, 30, 1)]  0                                            
__________________________________________________________________________________________________
convLayer (Conv2D)              (None, 26, 26, 128)  3328        input_4[0][0]                    
__________________________________________________________________________________________________
convLayer2 (Conv2D)             (None, 22, 22, 128)  409728      convLayer[0][0]                  
__________________________________________________________________________________________________
flatten_6 (Flatten)             (None, 900)          0           input_4[0][0]                    
_______________________________________________________________________________________

### Define procedure to calculate score

In [12]:

def CalcScores(ypred, ytruth):
  ydiff = ypred - ytruth
  score = np.sum( ydiff*ydiff )

  # The data sets were scaled by 1/amp_max when read in in order to force them
  # to be 0-1. This was not really needed since amp_max is set to 1. However,
  # in the event that I should change that above, I would need to scale up the
  # score by this value to match what the official scorer is doing. (At least
  # what I think it is doing.)
  score = score * amp_max*amp_max  # remember this is a squared sum!

  scale_fac = 2000/ypred.shape[0]  # scale to number of events in judge set
  corrected_score = score * scale_fac

  vals = {'score':score, 'corrected_score':corrected_score}

  return vals

### Calculate scores for both training and testing sets

In [13]:


ytrain_pred = model.predict(xtrain)
ytest_pred  = model.predict(xtest)

print('\nTRAIN:')
scores = CalcScores( ytrain_pred, ytrain )
print( scores )

print('\nTEST:')
scores = CalcScores( ytest_pred, ytest )
print( scores )

print('\nEstimated score: %f' % scores['corrected_score'])


TRAIN:
{'score': 196.39780586678452, 'corrected_score': 39.27956117335691}

TEST:
{'score': 58.72206616405455, 'corrected_score': 58.72206616405455}

Estimated score: 58.722066


### Calculate for judging and save to Google drive

In [14]:

yjudge_pred = model.predict(xjudge)

yjudge_pred = yjudge_pred*amp_max  # don't forget to rescale in case amp_max changes!

df = pd.DataFrame( yjudge_pred.reshape(yjudge_pred.shape[0], 900) )
df.to_csv('problem02_David Lawrence_002.csv', header=None, index = False)
#!cp 'problem02_David Lawrence_002.csv'  /gdrive/MyDrive/work/2021.07.29.hackathon